# SSYK 2012 Pipeline: Pull → Merge → Aggregate

**What this data is.** SCB (Statistics Sweden) publishes the number of employed persons broken down by occupation, sex, age group and year. Occupations follow **SSYK 2012** — the Swedish Standard Classification of Occupations, 2012 revision — which assigns every occupation a 4-digit code (e.g. `2142` = Engineering professionals in buildings and civil engineering). Each row in the raw data is the headcount for one specific (occupation × sex × age group × year) combination.

**What this pipeline does.** SCB releases the data in separate time-period tables rather than one continuous series. This pipeline pulls all relevant tables, deduplicates overlapping years, and rolls the 4-digit data up to coarser SSYK levels so analysis can happen at any level of detail.

| Stage | Script | Input | Output |
|---|---|---|---|
| **Pull** | `pull.py` | SCB API (AM0208) | `data/raw/ssyk12_*.parquet` |
| **Merge** | `merge.py` | `data/raw/ssyk12_*.parquet` | `data/processed/ssyk12_combined_cleaned.parquet` |
| **Aggregate** | `aggregate.py` | combined parquet + `structure_ssyk12.csv` | `data/processed/ssyk12_aggregated_ssyk4_to_ssyk1.parquet` |

The scripts in `scripts/` are the source of truth — this notebook imports from them directly and is meant to be run top-to-bottom to refresh the data.

In [ ]:
import sys
from pathlib import Path

import polars as pl

# Ensure project root is on sys.path so `scripts` package is importable
root = Path.cwd().resolve()
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

---
## Stage 1 — Pull

SCB does not offer the full employment series as a single download. Instead, it publishes overlapping tables that each cover a fixed window, updating the most recent window as new years become available. We pull all four:

| Table ID | SCB endpoint | Years | Note |
|---|---|---|---|
| `ssyk96_05_to_13` | AM0208 / Yreg34 | 2005–2013 | Uses the older **SSYK 96** classification — kept separate, not merged with the SSYK 2012 tables |
| `ssyk12_14_to_18` | AM0208 / YREG51 | 2014–2018 | First SSYK 2012 series |
| `ssyk12_19_to_21` | AM0208 / YREG51N | 2019–2021 | Overlap: years 2020–2021 also appear in the next table |
| `ssyk12_20_to_24` | AM0208 / YREG51BAS | 2020–2024 | Most recent series; its values take precedence for the overlapping years |

`fetch_clean_write` handles one table end-to-end: it discovers the variable keys from the API, builds a full-cube query, fetches the data, maps codes to labels, and writes a Parquet file. In production (`pull.py`) all four tables are fetched in parallel; here we run them sequentially so progress is visible.

In [ ]:
from scripts.pull import default_config as pull_config

cfg = pull_config()
print("OUT_DIR: ", cfg.out_dir)
print("Tables:  ", list(cfg.tables.keys()))

In [ ]:
from scripts.pull import fetch_clean_write

# Run sequentially here for visibility; pull.py uses ThreadPoolExecutor in production
pull_results: dict[str, bool] = {}
for tab_id, spec in cfg.tables.items():
    pull_results[tab_id] = fetch_clean_write(cfg, tab_id, spec)

ok = sum(pull_results.values())
print(f"\nPull complete: {ok}/{len(pull_results)} succeeded")
print(pull_results)

In [ ]:
# Quick look at one pulled file
sample_file = cfg.out_dir / "ssyk12_14_to_18.parquet"
if sample_file.exists():
    df_sample = pl.read_parquet(sample_file)
    print(f"Schema: {df_sample.schema}")
    print(f"Shape:  {df_sample.shape}")
    display(df_sample.head(5))

---
## Stage 2 — Merge

The three `ssyk12_*.parquet` files (note: `ssyk96` is excluded because it uses a different classification) contain overlapping data: years 2020 and 2021 appear in both `ssyk12_19_to_21` and `ssyk12_20_to_24`. Simply concatenating them would double-count those years.

The merge logic:
1. **Union** all three files, tagging each row with a `source_rank` (0 = newest file, i.e. `ssyk12_20_to_24`).
2. **Detect duplicates** — rows sharing the same identity key: `(code, occupation, age, year, sex)`.
3. **Resolve** by sorting on `source_rank` and keeping the first occurrence, so the newest published value always wins.
4. Drop the provenance columns and write the cleaned dataset.

In [ ]:
from scripts.merge import (
    PROVENANCE_COLS,
    get_id_cols,
    lazy_union,
    lf_rowcount,
    list_files_newest_first,
)
from scripts.merge import (
    default_config as merge_config,
)

mcfg = merge_config()
files = list_files_newest_first(mcfg.raw_dir)

print(f"Files found: {len(files)}")
for f in files:
    print(f"  {f.name}")

In [ ]:
first_cols = pl.read_parquet(files[0], n_rows=1).columns
id_cols = get_id_cols(first_cols)
print("Identity columns:", id_cols)

In [ ]:
combined_lf = lazy_union(files)
combined_rows = lf_rowcount(combined_lf)
print(f"Combined rows (pre-dedup): {combined_rows:,}")

# Duplicate stats
dup_rows = (
    combined_lf
    .with_columns(pl.len().over(id_cols).alias("_n"))
    .filter(pl.col("_n") > 1)
    .select(pl.len())
    .collect()
    .item()
)
print(f"Duplicate rows: {dup_rows:,}")

In [ ]:
# Resolve: keep newest (source_rank 0 = newest file)
cols_to_drop = [c for c in PROVENANCE_COLS if c in combined_lf.columns]
df_merged = (
    combined_lf
    .sort("source_rank")
    .unique(subset=id_cols, keep="first")
    .drop(cols_to_drop)
    .collect()
)

df_merged.write_parquet(mcfg.out_file)
print(f"Rows after dedup: {df_merged.height:,}")
print(f"Saved: {mcfg.out_file}")
display(df_merged.head(5))

---
## Stage 3 — Aggregate

The raw data is at **SSYK4** (4-digit) level — the most granular: ~430 distinct occupations. SSYK 2012 is a strict hierarchy, so every coarser level is just a prefix of the 4-digit code:

```
2142  →  214  →  21  →  2
(Engineering professionals       (Engineering     (Science &    (Professionals)
 in buildings/civil engineering)  professionals)   engineering)
```

`aggregate.py` slices the code column to derive `ssyk3`, `ssyk2`, and `ssyk1`, sums `count` within each group, and stacks the four levels into a single long-format dataset. It then joins `structure_ssyk12.csv` to attach a human-readable occupation name to every code at every level.

In [ ]:
from scripts.aggregate import (
    add_ssyk_levels,
    aggregate_all_levels,
    default_paths,
    diagnostics,
    load_name_map,
    load_ssyk4,
    map_occupation_names,
)

paths = default_paths()
print("IN_FILE: ", paths.in_file)
print("MAP_FILE:", paths.map_file)
print("OUT_FILE:", paths.out_file)

In [ ]:
df_ssyk4 = load_ssyk4(paths)
print(f"Schema: {df_ssyk4.schema}")
print(f"Shape:  {df_ssyk4.shape}")
display(df_ssyk4.head(5))

In [ ]:
lf = add_ssyk_levels(df_ssyk4)
df_all = aggregate_all_levels(lf)

print("Rows per level:")
display(
    df_all.group_by("level").agg(pl.len().alias("rows")).sort("level"),
)

### 3b. Map occupation names and check coverage

`structure_ssyk12.csv` contains a name for every SSYK code at all four levels. After joining it we run a quick diagnostic: `unmapped_rows = 0` means every code in the data has a matching name in the lookup file — nothing fell through. If codes were missing you'd see them listed so they can be added to the CSV.

In [ ]:
df_name_map = load_name_map(paths)
df_final = map_occupation_names(df_all, df_name_map)

diag, unmapped = diagnostics(df_final)
print("Mapping diagnostics:")
display(diag)

if unmapped.height > 0:
    print("Unmapped SSYK codes:")
    display(unmapped)

In [ ]:
df_final.write_parquet(paths.out_file)
print(f"Saved: {paths.out_file} ({df_final.height:,} rows)")

---
## Inspect Final Dataset

The output is a single long-format Parquet with 139 k rows covering all four SSYK levels, 2014–2024. Each row is the employed-person headcount for one `(level, ssyk_code, age, sex, year)` combination, with the occupation name attached.

Because all four levels live in the same file, you can filter to the granularity you need:
- `level == "SSYK1"` → 9 major occupation groups, good for broad trends
- `level == "SSYK4"` → ~430 specific occupations, good for detailed analysis

The `total_count` in the summary below double-counts men + women — use `sex` as a filter or grouping variable when computing population totals.

In [ ]:
df = pl.read_parquet(paths.out_file)
print(f"Schema: {df.schema}")
print(f"Shape:  {df.shape}")
display(df.head(10))

In [ ]:
# Row counts by level and year
display(
    df.group_by(["level", "year"])
    .agg(pl.col("count").sum().alias("total_count"))
    .sort(["level", "year"]),
)